# Сбор данных о школах России через OpenStreetMap (Overpass API)
Скачиваем все школы России по федеральным округам, сохраняем в CSV.

In [ ]:
import requests
import json
import pandas as pd
import time
import os

OUTPUT_PATH = '../data/raw/schools/schools_osm.csv'
OVERPASS_URL = 'https://overpass.kumi.systems/api/interpreter'

# Бounding boxes федеральных округов России [min_lat, min_lon, max_lat, max_lon]
FEDERAL_DISTRICTS = {
    'ЦФО (Центральный)':       (50.0, 31.0, 57.5, 41.5),
    'СЗФО (Северо-Западный)':  (54.0, 26.5, 70.0, 62.0),
    'ЮФО (Южный)':             (43.5, 36.5, 50.0, 47.0),
    'СКФО (Северо-Кавказский)':(43.0, 40.5, 47.0, 49.5),
    'ПФО (Приволжский)':       (51.0, 44.0, 62.0, 59.0),
    'УФО (Уральский)':         (54.0, 58.0, 72.0, 69.0),
    'СФО (Сибирский)':         (49.0, 68.0, 74.0, 99.0),
    'ДФО (Дальневосточный)':   (42.0, 99.0, 77.0, 141.0),
    'ДФО-восток (Чукотка)':    (60.0, 141.0, 77.0, 190.0),
}
print('Округа для скачивания:', len(FEDERAL_DISTRICTS))

In [ ]:
def fetch_schools(bbox, timeout=60, retries=3):
    """Скачать школы из Overpass API для заданного bounding box."""
    min_lat, min_lon, max_lat, max_lon = bbox
    query = f"""
    [out:json][timeout:{timeout}];
    (
      node["amenity"="school"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="school"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    for attempt in range(retries):
        try:
            resp = requests.post(OVERPASS_URL, data={'data': query}, timeout=timeout+10)
            resp.raise_for_status()
            data = resp.json()
            return data.get('elements', [])
        except Exception as e:
            print(f'  Попытка {attempt+1}/{retries} ошибка: {e}')
            time.sleep(10)
    return []


def parse_element(el):
    """Извлечь координаты и теги из элемента OSM."""
    tags = el.get('tags', {})
    # Для way берём center, для node — прямые координаты
    if el['type'] == 'way':
        lat = el.get('center', {}).get('lat')
        lon = el.get('center', {}).get('lon')
    else:
        lat = el.get('lat')
        lon = el.get('lon')
    return {
        'osm_id':   el.get('id'),
        'osm_type': el.get('type'),
        'lat':      lat,
        'lon':      lon,
        'name':     tags.get('name', ''),
        'addr_region':  tags.get('addr:region', ''),
        'addr_city':    tags.get('addr:city', ''),
        'addr_street':  tags.get('addr:street', ''),
        'addr_housenumber': tags.get('addr:housenumber', ''),
        'operator':     tags.get('operator', ''),
        'capacity':     tags.get('capacity', ''),
        'grades':       tags.get('grades', ''),
    }

print('Функции определены.')

In [ ]:
all_records = []

for district, bbox in FEDERAL_DISTRICTS.items():
    print(f'Скачиваю: {district} ...', end=' ')
    elements = fetch_schools(bbox)
    records = [parse_element(el) for el in elements if el.get('lat') or el.get('center')]
    all_records.extend(records)
    print(f'{len(records)} школ')
    time.sleep(5)  # пауза между запросами

print(f'\nИтого: {len(all_records)} объектов')

In [ ]:
df = pd.DataFrame(all_records)

# Убрать дубликаты (могут появиться на границах округов)
df = df.drop_duplicates(subset='osm_id')

# Убрать записи без координат
df = df.dropna(subset=['lat', 'lon'])

# Фильтр: только территория России (примерный bbox)
df = df[(df['lat'].between(41, 82)) & (df['lon'].between(19, 192))]

print(f'После очистки: {len(df)} школ')
print(df.head())

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f'Сохранено: {OUTPUT_PATH}')

In [ ]:
# Быстрая статистика
print('=== Статистика ===' )
print(f'Всего школ: {len(df)}')
print(f'С названием: {df["name"].notna().sum()}')
print(f'С адресом региона: {(df["addr_region"] != "").sum()}')
print(f'С адресом города: {(df["addr_city"] != "").sum()}')
print()
print('Топ-10 регионов по количеству школ:')
print(df['addr_region'].value_counts().head(10))